Just copy and paste the example code of BI from the assignment

In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

In [ ]:
# Use matplotlib only; do not set custom colors/styles per instructions.
plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(42)

def poisson_pmf_trunc(lam=5.0, d_max=20):
    """Return normalized probabilities for D=0..d_max and the leftover (truncation) mass."""
    probs = np.array([math.exp(-lam) * lam**d / math.factorial(d) for d in range(d_max+1)], dtype=float)
    mass = probs.sum()
    if mass <= 0:
        probs = np.zeros(d_max+1, dtype=float)
        probs[0] = 1.0
        mass = 1.0
    return probs / mass, 1.0 - mass


def build_dp(
    T=12,      # number of hours (stages)
    x_max=70,  # maximum queue length (state)
    lam=5.0,   # Poisson arrival rate per hour
    d_max=20,  # truncate Poisson support to {0..d_max}
    w=200.0,   # waiting cost per patient per hour
    W=400.0,   # terminal cost per patient
    p=500.0,   # on-demand doctor wage per hour
    u_max=None # optional cap on action (if None, compute a safe cap per state)
):
    """
    Solve the DP by backward induction with expectation over Poisson arrivals.
    Returns:
        J (np.ndarray): shape (T+2, x_max+1) cost-to-go; index t in [1..T+1], x in [0..x_max].
        U (np.ndarray): shape (T+1, x_max+1) optimal action table; U[t,x] is u*_t(x).
    """
    pdist, _ = poisson_pmf_trunc(lam=lam, d_max=d_max)
    D_support = np.arange(d_max+1, dtype=int)

    J = np.full((T+2, x_max+1), np.inf, dtype=float)  # cost-to-go
    U = np.zeros((T+1, x_max+1), dtype=int)          # optimal action

    # Terminal cost
    J[T+1, :] = W * np.arange(x_max+1, dtype=float)

    # Backward induction
    for t in range(T, 0, -1):
        for x in range(x_max+1):
            # Dynamic action cap: ensure we don't search an excessive range
            if u_max is None:
                # enough to clear worst-case (x + d_max) with 20 + 2u capacity
                u_cap = max(0, math.ceil((x + d_max - 20) / 2))
                u_cap = min(u_cap, 60)  # hard cap
            else:
                u_cap = u_max

            best_val = np.inf
            best_u = 0
            for u in range(u_cap + 1):
                # YOUR CODE HERE
                # compute total cost = immediate cost + expected future cost
                imm = ... # YOUR CODE HERE immediate cost is the wait cost and hiring new surge costs
                cap = 2 * (10 + u) # What is this?

                x_next = x + D_support - cap
                x_next = np.clip(x_next, 0, x_max)

                exp_future = ... # YOUR CODE HERE Calculate the expected futrue cost, you may need to multiply two vectors
                total = ... # YOUR CODE HERE Obtain the total cost
                # End YOUR CODE HERE

                if total < best_val:
                    best_val = total
                    best_u = u

            J[t, x] = best_val
            U[t, x] = best_u

    return J, U

In [ ]:
T = 12
X_MAX = 70
LAM = 17.0
D_MAX = 20
W = 400.0
w = 30.0
WAGE = 500.0
J, U = build_dp(T=T, x_max=X_MAX, lam=LAM, d_max=D_MAX, w=w, W=W, p=WAGE)

In [ ]:
states = np.arange(X_MAX+1)
hours  = np.arange(1, T+1)

BI_policy_df = pd.DataFrame(U[1:T+1, :].T, index=states, columns=hours)  # shape (x, t)
BI_cost_df   = pd.DataFrame(J[1:T+1, :].T, index=states, columns=hours)  # shape (x, t)

# Show heads to confirm structure
display(BI_policy_df.head(3))
display(BI_cost_df.head(3))